In [ ]:
from SteerEnergyStorage.Formulations.ElectrodeFormulations import ElectrodeFormulation
from SteerEnergyStorage.Constructions.Electrodes import Cathode, Anode
from SteerEnergyStorage.Formulations.Stacks import Stack
from SteerEnergyStorage.Constructions.Cells import StackedPrismaticCell
from SteerEnergyStorage.Materials.ElectrodeMaterials import CathodeMaterial, AnodeMaterial, Binder, ConductiveAdditive
from SteerEnergyStorage.Materials.CurrentCollectors import CurrentCollector
from SteerEnergyStorage.Materials.Separators import Separator
from SteerEnergyStorage.Materials.Electrolytes import Electrolyte
from SteerEnergyStorage.Constructions.Containers import PrismaticCase, PrismaticLid, PrismaticShell

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

In [ ]:
cathode_n = 40
anode_n = 40
cathode_mass_loading_range = np.linspace(8, 15, cathode_n)
anode_mass_loading_range = np.linspace(8, 15, anode_n)

In [ ]:
# construct cathodes

cathodes = []

for cml in cathode_mass_loading_range:
    for aml in anode_mass_loading_range:

        # create formulation materials
        cathode_active_material = CathodeMaterial(name=f"Faradion_Gen2_4.25V", specific_cost=11.26, density=4)
        cathode_conductive_additive = ConductiveAdditive(specific_cost=9, density=1.9)
        cathode_binder = Binder(specific_cost=15, density = 1.7)

        # create formulation with noisy fractions
        active_material_fraction = np.random.normal(89, 1)
        binder_fraction = np.random.normal(5, 1)
        conductive_additive_fraction = 100 - active_material_fraction - binder_fraction

        cathode_formulation = ElectrodeFormulation(active_materials={cathode_active_material: active_material_fraction},
                                                   binders={cathode_binder: binder_fraction},
                                                   conductive_additives={cathode_conductive_additive: conductive_additive_fraction}
                                                   )

        # create current collector
        cathode_current_collector = CurrentCollector(formula="Al", thickness=15, length=15, width=11.8, bare_tab_area=16)

        # create cathode
        cathode = Cathode(formulation=cathode_formulation,
                          mass_loading=cml,
                          current_collector=cathode_current_collector,
                          calender_density=2.6
                          )
        
        # append cathode to list
        cathodes.append(cathode) 


In [ ]:
# construct anode

anodes = []

for cml in cathode_mass_loading_range:
    for aml in anode_mass_loading_range:
        # create formulation materials
        anode_active_material = AnodeMaterial(name="faradion_hc", specific_cost=14.27, density=1.5)
        anode_conductive_additive = ConductiveAdditive(specific_cost=9, density=1.9)
        anode_binder = Binder(specific_cost=10, density=1.7)

        # create formulation with noisy fractions
        active_material_fraction = np.random.normal(88, 1)
        binder_fraction = np.random.normal(3, 1)
        conductive_additive_fraction = 100 - active_material_fraction - binder_fraction

        # create formulation
        anode_formulation = ElectrodeFormulation(active_materials={anode_active_material: active_material_fraction},
                                                binders={anode_binder: binder_fraction},
                                                conductive_additives={anode_conductive_additive: conductive_additive_fraction})

        # create current collector
        anode_current_collector = CurrentCollector(formula="Cu", thickness=4, length=15, width=11.8, bare_tab_area=7.55)

        # create anode
        anode = Anode(formulation=anode_formulation,
                      mass_loading=aml,
                      current_collector=anode_current_collector,
                      calender_density=0.85)
        
        anodes.append(anode)


In [ ]:
# Other cell components

seperators = []
stacks = []
for cml in cathode_mass_loading_range:
    for aml in anode_mass_loading_range:
        # construct seperator
        seperator = Separator(thickness=16, areal_cost=0.9, density=0.4, slit_width=12, porosity=47, fold_length=18.6)
        seperators.append(seperator)


In [ ]:

# make electrolyte
electrolyte = Electrolyte(name="LiPF6", specific_cost=8.94, density=1.2)

cases = []
stacks = []

for cathode, anode, seperator in zip(cathodes, anodes, seperators):

        prismatic_lid = PrismaticLid(cost=0.05, mass=10, external_width=1.3, internal_width=0.8)
        prismatic_shell = PrismaticShell(cost=0.15, mass=60, internal_length=19, internal_width=13, internal_height=0.5, wall_thickness=0.5)
        prismatic_case = PrismaticCase(lid=prismatic_lid, shell=prismatic_shell)
        cases.append(prismatic_case)

        stack = prismatic_case.get_optimized_stack(cathode=cathode, anode=anode, separator=seperator)
        
        stacks.append(stack)


In [ ]:
cells = []
for stack, case in zip(stacks, cases):
        
        cell = StackedPrismaticCell(stack=stack,
                                    prismatic_case=case,
                                    electrolyte=electrolyte,
                                    electrolyte_overfill=10,
                                    reversible_capacity=4.5,
                                    irreversible_capacity=1)
    
        cells.append(cell)

In [ ]:
figure = cells[0].get_mass_breakdown_plot(width=900, height=400)
figure.show()

In [ ]:
figure = cells[0].get_cost_breakdown_plot(width=900, height=400)
figure.show()

In [ ]:
# plot energy densities

plot_data = pd.DataFrame({
    'cathode_mass_loading': [c.stacks[0].cathodes[0].mass_loading for c in cells],
    'anode_mass_loading': [c.stacks[0].anodes[0].mass_loading for c in cells],
    'energy_density': [c.energy_density for c in cells]
})

fig = go.Figure()
fig.add_trace(go.Contour(x=plot_data['cathode_mass_loading'],
                         y=plot_data['anode_mass_loading'],
                         z=plot_data['energy_density'],
                         colorscale='rdbu',
                         ncontours=2000,
                         contours_showlines=False
                         )
              )
fig.update_layout(title='Energy Density vs. Mass Loading',
                  xaxis_title='Cathode Mass Loading (mg/cm^2)',
                  yaxis_title='Anode Mass Loading (mg/cm^2)',
                  xaxis=dict(tickmode='linear', tick0=5, dtick=2),
                  yaxis=dict(tickmode='linear', tick0=5, dtick=2),
                  width=600,
                  height=600
                  )
fig.show()


In [ ]:
# plot energy densities

plot_data = pd.DataFrame({
    'cathode_mass_loading': [c.stacks[0].cathodes[0].mass_loading for c in cells],
    'anode_mass_loading': [c.stacks[0].anodes[0].mass_loading for c in cells],
    'normalised_cost': [c.normalized_cost for c in cells]
})

fig = go.Figure()
fig.add_trace(go.Contour(x=plot_data['cathode_mass_loading'],
                         y=plot_data['anode_mass_loading'],
                         z=plot_data['normalised_cost'],
                         colorscale='rdbu',
                         ncontours=2000,
                         contours_showlines=False
                         )
              )
fig.update_layout(title='Normalised Cost ($/kWh) vs. Mass Loading',
                  xaxis_title='Cathode Mass Loading (mg/cm^2)',
                  yaxis_title='Anode Mass Loading (mg/cm^2)',
                  xaxis=dict(tickmode='linear', tick0=5, dtick=2),
                  yaxis=dict(tickmode='linear', tick0=5, dtick=2),
                  width=600,
                  height=600
                  )

fig.show()

In [ ]:
# plot energy densities

plot_data = pd.DataFrame({
    'cathode_mass_loading': [c.stacks[0].cathodes[0].mass_loading for c in cells],
    'anode_mass_loading': [c.stacks[0].anodes[0].mass_loading for c in cells],
    'stack_layers': [c.stacks[0]._n_layers for c in cells]
})

fig = go.Figure()
fig.add_trace(go.Contour(x=plot_data['cathode_mass_loading'],
                         y=plot_data['anode_mass_loading'],
                         z=plot_data['stack_layers'],
                         colorscale='rdbu',
                         ncontours=200,
                         contours_showlines=False
                         )
              )
fig.update_layout(title='Number of Stack Layers vs. Mass Loading',
                  xaxis_title='Cathode Mass Loading (mg/cm^2)',
                  yaxis_title='Anode Mass Loading (mg/cm^2)',
                  xaxis=dict(tickmode='linear', tick0=5, dtick=2),
                  yaxis=dict(tickmode='linear', tick0=5, dtick=2),
                  width=600,
                  height=600
                  )
fig.show()